# A Party Scheduling Problem with FICO Xpress
[![party1.ipynb](https://img.shields.io/badge/github-%23121011.svg?logo=github)](https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Open In Deepnote](https://deepnote.com/buttons/launch-in-deepnote-small.svg)](https://deepnote.com/launch?url=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Open In Gradient](https://assets.paperspace.io/img/gradient-badge.svg)](https://console.paperspace.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb) [![Powered by AMPL](https://h.ampl.com/https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/scheduling/party1/party1.ipynb)](https://ampl.com)

Description: A scheduling problem for visitor-host assignments. Feasibility version (no objective function). Demonstrates high-level modeling in AMPL MP, AMPL Python API, and tuning in FICO Xpress

Tags: AMPLPY, MP, highlights, scheduling, assignment, feasibility problem, tuning, xpress

Notebook author: Gleb Belov <<gleb@ampl.com>>

Reference:

1. AMPL Xpress documentation. [https://dev.ampl.com/solvers/xpress/index.html](https://dev.ampl.com/solvers/xpress/index.html).
2. AMPL MP Library: modeling guide. [https://mp.ampl.com/](https://mp.ampl.com/).

In [1]:
# Install dependencies
%pip install -q amplpy pandas

In [2]:
# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=["xpress"],  # modules to install
    license_uuid="default",  # license to use
)  # instantiate AMPL object and register magics

## Use `%%ampl_eval` to evaluate AMPL commands and declarations

In [3]:
%%ampl_eval

param B > 0, integer;
set BOATS := 1 .. B;
set HOSTS within BOATS;
set VISITORS := BOATS diff HOSTS;

param capacity {BOATS} integer >= 0;
param crew {BOATS} integer > 0;
param guest_cap {i in BOATS} := capacity[i] less crew[i];

param T > 0, integer;
set TIMES := 1..T;

set MEET := {j in VISITORS, jj in VISITORS: j < jj};

       # (j,jj) in MEET if j and jj can meet -- each pair only once

var H {VISITORS,TIMES} integer, >= 1, <= B;  # host (boat) visited by j at t


VisitHosts {j in VISITORS, jj in VISITORS, t in TIMES}:
   H[j,t] != jj;

       # a visitor may not serve as a host: H[j,t] in VISITORS

subj to VisitOnce {j in VISITORS}:
   alldiff {t in TIMES} H[j,t];

       # a crew may visit each host at most once

subj to Cap {i in HOSTS, t in TIMES}: 
   sum {j in VISITORS} (if H[j,t] = i then crew[j]) <= guest_cap[i];

       # boats may not have more visitors than they can handle

subj to MeetDefn {(j,jj) in MEET}:
   atmost 1 {t in TIMES} (H[j,t] = H[jj,t]);

       # two crews may meet at most once


## Define instance data in Python

In [4]:
import pandas as pd

# For each of 26 boats, crew size
crew_size = [2,	2,	2,	2,	4,	4,	4,	1,	2,	2,	2,	3,	4,	2,	3,	6,	2,	2,	4,	2,	4,	5,	4,	4,	2,	2]
# Which boats are hosts
hosts = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 16, 20]
# Boat capacity
capacity = [6,	8,	12,	12,	12,	12,	12,	10,	10,	10,	10,	10,	8,	8,	8,	12,	8,	8,	8,	8,	8,	8,	7,	7,	7,	7]
# Time horizon
time_horizon = 6

## Load data directly from Python data structures using [amplpy](https://amplpy.readthedocs.io/)

In [5]:
ampl.param['B'] = len(crew_size)
ampl.set['HOSTS'] = hosts
ampl.param['capacity'] = capacity
ampl.param['crew'] = crew_size
ampl.param['T'] = time_horizon

## Solve with FICO Xpress

We solve the instance with default options.

In [6]:
ampl.option["solver"] = "xpress"
ampl.option["mp_options"] = "outlev=1"

ampl.snapshot('party1_snapshot.run')

ampl.solve()

XPRESS 9.8.0 (46.01.01):   tech:outlev = 1

AMPL MP initial flat model has 78 variables (78 integer, 0 binary);
No objectives;
Constraints:  156 linear;
Algebraic expressions:  78 count; 1014 ifthen;
Logical expressions:  2028 conditional (in)equalitie(s); 13 alldiff;

AMPL MP final model has 4777 variables (2197 integer, 2574 binary);
No objectives;
Constraints:  1118 linear;
Logical expressions:  468 indge; 468 indle;


FICO Xpress v9.8.0, Hyper, solve started 11:09:10, Feb 27, 2026
Heap usage: 2180KB (peak 2180KB, 54KB system)
Minimizing MILP  using up to 8 threads and up to 16GB memory, with default controls
Original problem has:
      2054 rows         4777 cols         9672 elements      2574 entities
       936 inds
Presolved problem has:
      1884 rows         2490 cols         8790 elements      2490 entities
LP relaxation tightened
Presolve finished in 0 seconds
Heap usage: 4403KB (peak 6108KB, 54KB system)

Coefficient range                    original                 solve

## Check solution in AMPL

The below AMPL code checks solution for correctness. Additional Python code can be used for independent checking.

In [7]:
%%ampl_eval
display solve_result, solve_result_num;

display min {i in 1.._ncons} _con[i].slack;
display min {i in 1.._nlogcons} _logcon[i].val;

display _conname, _con.slack;
display _logconname, _logcon.val;

solve_result = solved
solve_result_num = 0

min{i in 1 .. _ncons} _con[i].slack = 0

min{i in 1 .. _nlogcons} _logcon[i].val = 1

:     _conname   _con.slack    :=
1    'Cap[1,1]'        0
2    'Cap[1,2]'        2
3    'Cap[1,3]'        4
4    'Cap[1,4]'        0
5    'Cap[1,5]'        4
6    'Cap[1,6]'        0
7    'Cap[2,1]'        0
8    'Cap[2,2]'        4
9    'Cap[2,3]'        6
10   'Cap[2,4]'        1
11   'Cap[2,5]'        4
12   'Cap[2,6]'        4
13   'Cap[3,1]'       10
14   'Cap[3,2]'        8
15   'Cap[3,3]'        6
16   'Cap[3,4]'        2
17   'Cap[3,5]'        3
18   'Cap[3,6]'        7
19   'Cap[4,1]'        3
20   'Cap[4,2]'        4
21   'Cap[4,3]'        8
22   'Cap[4,4]'        3
23   'Cap[4,5]'        9
24   'Cap[4,6]'        8
25   'Cap[5,1]'        2
26   'Cap[5,2]'        8
27   'Cap[5,3]'        6
28   'Cap[5,4]'        8
29   'Cap[5,5]'        8
30   'Cap[5,6]'        4
31   'Cap[6,1]'        3
32   'Cap[6,2]'        3
33   'Cap[6,3]'        8
34   'Cap[6

## Retrieve solution as a Pandas dataframe

In [8]:
ampl.var["H"].to_pandas()

H.val
index0 index1       
8      1          12
       2           7
       3          20
       4          11
       5           4
...              ...
26     2           1
       3           4
       4          11
       5           9
       6           2

[78 rows x 1 columns]

## Automatic tuning of solver parameters

Let us apply tuner and compare runtimes before and after.

In [9]:
ampl.option["reset_initial_guesses"] = 1   # Don't send the known solution as the initial guess

ampl.option["mp_options"] = \
        "outlev=1 " + \
        "tech:xprs_tunermode=1 " + \
        "tech:xprs_tunermethod=1 " + \
        "tech:xprs_tunerthreads=8 " + \
        "tech:xprs_tunermaxtime=180" + \
        "tech:xprs_tunerhistory=0"

ampl.solve()

XPRESS 9.8.0 (46.01.01):   tech:outlev = 1
  tech:xprs_tunermode = 1
  tech:tunermethod = 1
  tech:tunerthreads = 8
  tech:tunetimelim = 180
  tech:tunerhistory = 0

AMPL MP initial flat model has 78 variables (78 integer, 0 binary);
No objectives;
Constraints:  156 linear;
Algebraic expressions:  78 count; 1014 ifthen;
Logical expressions:  2028 conditional (in)equalitie(s); 13 alldiff;

AMPL MP final model has 4777 variables (2197 integer, 2574 binary);
No objectives;
Constraints:  1118 linear;
Logical expressions:  468 indge; 468 indle;


Tuner: tuning MIP problem UnknownProblem
Tuner: testing up to 8 control settings in parallel
baseline setting: (none)

Tuner: starting adaptive phase 1: trying each control
   RunID Stat     Solution        Bound     Integral     Gap  RunTime TotTime
*      4    S      .000000      .000000     1.004141    0.00%    1.02       1 - COVERCUTS=0
setting:
  COVERCUTS            = 0
logfile: UnknownProblem-20260227.110911-task0004.log

       7    S      

# Conclusion

Automatic tuning improved single-solve runtime using the new setting `COVERCUTS=2;CUTFACTOR=0.5;HEURSEARCHROOTCUTFREQ=2;NODEPROBINGEFFORT=0;PRESOLVE=0;SCALING=16`. See [https://dev.ampl.com/solvers/xpress/index.html](https://dev.ampl.com/solvers/xpress/index.html) for a full list of AMPL Xpress options.